In [1]:
import time
from aic_gym.envs.base_classes import Integrator
import yaml
import gym
import numpy as np
from argparse import Namespace

from numba import njit

from pyglet.gl import GL_POINTS

In [5]:
yaml_file_path = "/home/vesta-22/aichallenge_gym/gym/f110_gym/envs/maps/gray_aichallenge"
env = gym.make('aic_gym:aic-v0', map=yaml_file_path, map_ext='.png', num_agents=1, time_step=0.1, integrator=Integrator.RK4)

/home/vesta-22/aichallenge-2025/aichallenge/workspace/src/aichallenge_gym/gym/aic_gym/envs/base_classes.py:95: UserWarning: Chosen integrator is RK4. This is different from previous versions of the gym.
  warnings.warn(f"Chosen integrator is RK4. This is different from previous versions of the gym.")


In [2]:
yaml_file_path = "/home/vesta-22/aichallenge_gym/gym/f110_gym/envs/maps/gray_aichallenge"

custom_params = {
    # ユーザー指定のパラメータ
    'm': 160.0,                               # 質量 [kg]
    'length': 2.0,                            # 全長 [m]
    'width': 1.45,                            # 全幅 [m]
    'lf': 1.087/2.0,                        # 重心から前軸までの距離 [m] (ホイールベースの半分と仮定)
    'lr': 1.087/2.0,                        # 重心から後軸までの距離 [m] (ホイールベースの半分と仮定)
    's_max': 0.64,         # 最大ステアリング角 [rad]
    's_min': -0.64,        # 最小ステアリング角 [rad]
    'a_max': 3.2,                             # 最大加速度 [m/s^2]
    
    # デフォルトまたは推定値を使用するパラメータ
    'mu': 1.0489,                             # 路面摩擦係数 (デフォルト値)
    'C_Sf': 5.64718,                            # 前輪コーナリングスティフネス (デフォルト値)
    'C_Sr': 5.65456,                           # 後輪コーナリングスティフネス (デフォルト値)
    'h': 0.2,                                 # 重心の高さ [m] (推定値)
    'I': 81.37,           # 慣性モーメント [kgm^2] (均一な棒として概算)
    'sv_min': -0.32,                          # 最小ステアリング速度 [rad/s] (デフォルト値)
    'sv_max': 0.32,                            # 最大ステアリング速度 [rad/s] (デフォルト値)
    'v_switch': 7.319,                        # 速度スイッチ (デフォルト値)
    'v_min': -10.0,                           # 最小速度 [m/s] (調整値)
    'v_max': 25.0,                            # 最大速度 [m/s] (調整値)
}

env = gym.make('aic_gym:aic-v0', map=yaml_file_path, map_ext='.png', num_agents=1, time_step=0.1, integrator=Integrator.RK4, params=custom_params)


def render_callback(env_renderer):
    # custom extra drawing function

    e = env_renderer

    # update camera to follow car
    x = e.cars[0].vertices[::2]
    y = e.cars[0].vertices[1::2]
    top, bottom, left, right = max(y), min(y), min(x), max(x)

    e.left = left - 1200
    e.right = right + 1200
    e.top = top + 1200
    e.bottom = bottom - 1200

env.add_render_callback(render_callback)

# # test render
# # 環境をリセット
# obs = env.reset(np.array([[89653.7, 43122.5, 3.92699]]))
# # obs = env.reset(np.array([[0, 0, 0]]))

# # 手動で車を動かすためのループ
# for i in range(10000):  # 100ステップ実行
#     # ランダムなアクションを生成（スロットル、ステアリング）
#     speed = np.random.uniform(0, 180)
#     steer = np.random.uniform(-1, 1)
#     action = np.array([steer, speed])  
    
#     # 環境をステップ実行
#     obs, reward, done, info = env.step(np.array([action]))
    
#     # 環境をレンダリング
#     env.render(mode='human')
    
#     # 終了条件
#     if done:
#         print(f"Episode finished after {i+1} steps")
#         break

# # 環境を閉じる
# env.close()


/home/vesta-22/aichallenge-2025/aichallenge/workspace/src/aichallenge_gym/gym/aic_gym/envs/base_classes.py:95: UserWarning: Chosen integrator is RK4. This is different from previous versions of the gym.
  warnings.warn(f"Chosen integrator is RK4. This is different from previous versions of the gym.")


In [3]:
waypoint = np.loadtxt("/home/vesta-22/aichallenge_gym/examples/raceline_awsim_15km_pp.csv", delimiter=",", skiprows=1)

print(waypoint.shape) # x,y,z,x_quat,y_quat,z_quat,w_quat,speed

# chage x, y, theta, speed

# x, y, theta, speed の配列を作成
# thetaはクォータニオンからヨー角（theta）を計算
def quat_to_yaw(z, w):
    # 2D平面用のz, wクォータニオンからヨー角を計算
    return 2 * np.arctan2(z, w)

x = waypoint[:, 0]
y = waypoint[:, 1]
z_quat = waypoint[:, 5]
w_quat = waypoint[:, 6]
speed = waypoint[:, 7]
theta = quat_to_yaw(z_quat, w_quat)

# 新しい配列を作成: x, y, theta, speed
waypoint_xythv = np.stack([x, y, theta, speed], axis=1)




(121, 8)


In [4]:
import numpy as np
from numba import njit

@njit(fastmath=False, cache=True)
def nearest_point_on_trajectory(point, trajectory):
    diffs = trajectory[1:,:] - trajectory[:-1,:]
    l2s   = diffs[:,0]**2 + diffs[:,1]**2
    dots = np.empty((trajectory.shape[0]-1, ))
    for i in range(dots.shape[0]):
        dots[i] = np.dot((point - trajectory[i, :]), diffs[i, :])
    t = dots / l2s
    t[t<0.0] = 0.0
    t[t>1.0] = 1.0
    projections = trajectory[:-1,:] + (t*diffs.T).T
    dists = np.empty((projections.shape[0],))
    for i in range(dists.shape[0]):
        temp = point - projections[i]
        dists[i] = np.sqrt(np.sum(temp*temp))
    min_dist_segment = np.argmin(dists)
    return projections[min_dist_segment], dists[min_dist_segment], t[min_dist_segment], min_dist_segment

@njit(fastmath=False, cache=True)
def first_point_on_trajectory_intersecting_circle(point, radius, trajectory, t=0.0, wrap=False):
    start_i = int(t)
    start_t = t % 1.0
    first_t = None
    first_i = None
    first_p = None
    trajectory = np.ascontiguousarray(trajectory)
    for i in range(start_i, trajectory.shape[0]-1):
        start = trajectory[i,:]
        end = trajectory[i+1,:]+1e-6
        V = np.ascontiguousarray(end - start)
        a = np.dot(V,V)
        b = 2.0*np.dot(V, start - point)
        c = np.dot(start, start) + np.dot(point,point) - 2.0*np.dot(start, point) - radius*radius
        discriminant = b*b-4*a*c
        if discriminant < 0:
            continue
        discriminant = np.sqrt(discriminant)
        t1 = (-b - discriminant) / (2.0*a)
        t2 = (-b + discriminant) / (2.0*a)
        if i == start_i:
            if t1 >= 0.0 and t1 <= 1.0 and t1 >= start_t:
                first_t = t1
                first_i = i
                first_p = start + t1 * V
                break
            if t2 >= 0.0 and t2 <= 1.0 and t2 >= start_t:
                first_t = t2
                first_i = i
                first_p = start + t2 * V
                break
        elif t1 >= 0.0 and t1 <= 1.0:
            first_t = t1
            first_i = i
            first_p = start + t1 * V
            break
        elif t2 >= 0.0 and t2 <= 1.0:
            first_t = t2
            first_i = i
            first_p = start + t2 * V
            break
    if wrap and first_p is None:
        for i in range(-1, start_i):
            start = trajectory[i % trajectory.shape[0],:]
            end = trajectory[(i+1) % trajectory.shape[0],:]+1e-6
            V = end - start
            a = np.dot(V,V)
            b = 2.0*np.dot(V, start - point)
            c = np.dot(start, start) + np.dot(point,point) - 2.0*np.dot(start, point) - radius*radius
            discriminant = b*b-4*a*c
            if discriminant < 0:
                continue
            discriminant = np.sqrt(discriminant)
            t1 = (-b - discriminant) / (2.0*a)
            t2 = (-b + discriminant) / (2.0*a)
            if t1 >= 0.0 and t1 <= 1.0:
                first_t = t1
                first_i = i
                first_p = start + t1 * V
                break
            elif t2 >= 0.0 and t2 <= 1.0:
                first_t = t2
                first_i = i
                first_p = start + t2 * V
                break
    return first_p, first_i, first_t

@njit(fastmath=False, cache=True)
def get_actuation(pose_theta, lookahead_point, position, lookahead_distance, wheelbase):
    waypoint_y = np.dot(np.array([np.sin(-pose_theta), np.cos(-pose_theta)]), lookahead_point[0:2]-position)
    speed = lookahead_point[2]
    if np.abs(waypoint_y) < 1e-6:
        return speed, 0.
    radius = 1/(2.0*waypoint_y/lookahead_distance**2)
    steering_angle = np.arctan(wheelbase/radius)
    return speed, steering_angle

class PurePursuitPlannerNotebook:
    def __init__(self, waypoints_xythv, wheelbase=0.33, vgain=1.0, lookahead_distance=1.0):
        self.waypoints = waypoints_xythv  # (N, 4): x, y, theta, speed
        self.wheelbase = wheelbase
        self.vgain = vgain
        self.lookahead_distance = lookahead_distance
        self.max_reacquire = 20.0

    def _get_current_waypoint(self, position, theta):
        wpts = self.waypoints[:, :2]
        nearest_point, nearest_dist, t, i = nearest_point_on_trajectory(position, wpts)
        if nearest_dist < self.lookahead_distance:
            lookahead_point, i2, t2 = first_point_on_trajectory_intersecting_circle(position, self.lookahead_distance, wpts, i+t, wrap=True)
            if i2 is None:
                return None
            current_waypoint = np.empty((3, ))
            current_waypoint[0:2] = wpts[i2, :]
            current_waypoint[2] = self.waypoints[i, 3]  # speed
            return current_waypoint
        elif nearest_dist < self.max_reacquire:
            return np.append(wpts[i, :], self.waypoints[i, 3])
        else:
            return None

    def plan(self, pose_x, pose_y, pose_theta):
        position = np.array([pose_x, pose_y])
        lookahead_point = self._get_current_waypoint(position, pose_theta)
        if lookahead_point is None:
            return 4.0, 0.0
        speed, steering_angle = get_actuation(pose_theta, lookahead_point, position, self.lookahead_distance, self.wheelbase)
        speed = self.vgain * speed
        return speed, steering_angle

In [5]:
# 例: PurePursuitPlannerNotebookのインスタンス化
planner = PurePursuitPlannerNotebook(waypoint_xythv, wheelbase=2.0, vgain=1.0, lookahead_distance=5.0)
obs, _, _, _ = env.reset(np.array([[89653.7, 43122.5, 3.92699]]))
# 環境ループ内で
for i in range(10000):
    speed, steer = planner.plan(obs['poses_x'][0], obs['poses_y'][0], obs['poses_theta'][0])
    action = np.array([[steer, speed]])
    obs, reward, done, info = env.step(action)
    env.render(mode='human')
    if done:
        break

In [ ]:
waypoint = np.loadtxt("/home/vesta-22/aichallenge_gym/examples/centerline_15km.csv", delimiter=",", skiprows=1)

print(waypoint.shape) # x,y,z,x_quat,y_quat,z_quat,w_quat,speed

# chage x, y, theta, speed

# x, y, theta, speed の配列を作成
# thetaはクォータニオンからヨー角（theta）を計算
def quat_to_yaw(z, w):
    # 2D平面用のz, wクォータニオンからヨー角を計算
    return 2 * np.arctan2(z, w)

x = waypoint[:, 0]
y = waypoint[:, 1]
z_quat = waypoint[:, 5]
w_quat = waypoint[:, 6]
speed = waypoint[:, 7]
theta = quat_to_yaw(z_quat, w_quat)

# 新しい配列を作成: x, y, theta, speed
waypoint_xythv = np.stack([x, y, theta, speed], axis=1)




(137, 8)
変換後のwaypoint shape: (137, 4)
先頭5行:
 [[8.96304488e+04 4.31309470e+04 2.09887078e+00 4.16666667e+00]
 [8.96290488e+04 4.31333470e+04 2.09887078e+00 4.16666667e+00]
 [8.96276488e+04 4.31357470e+04 2.21429744e+00 4.16666667e+00]
 [8.96258488e+04 4.31381470e+04 2.15879893e+00 4.16666667e+00]
 [8.96242488e+04 4.31405470e+04 2.09887078e+00 4.16666667e+00]]
